In [1]:
# Parameters
summary_config = {"run_run_comparison": False, "run_RTP_summary": False, "run_validation": True, "run_network_validation": False, "summary_list": {"RTP-summary-notebook": ["RTP_index", "RTP_congestion", "RTP_topsheet", "RTP_MIC", "RTP_person", "RTP_household", "RTP_access", "RTP_costs", "RTP_walk_bike", "RTP_emissions", "RTP_mode_share", "RTP_freight", "RTP_transit"], "activitysim-validation-notebook": ["park_and_ride"], "daysim-validation-notebook": ["all_tour_mode", "all_trip_mode", "auto_ownership", "day_pattern", "escort_tour_mode", "escort_trip_mode", "households", "intermediate_stop_generation", "other_home_based_tour_mode", "other_home_based_trip_mode", "persons", "school_location", "school_tour_mode", "school_trip_mode", "telecommute", "time_choice", "tours", "tour_destination", "transit_pass_ownership", "trips", "trip_destination", "workbased_subtour_generation", "workbased_subtour_mode", "work_location", "work_tour_mode", "work_trip_mode"], "network-validation-notebook": ["supplementals", "transit_validation", "traffic_validation", "link_analysis"], "run-comparison-notebook": ["topsheet", "population", "parking", "vmt", "transit"]}, "p_output_dir": "outputs/summary", "output_folder": "outputs", "survey_folder": "inputs/base_year/survey", "uncloned_folder": "uncloned", "sc_run_name": "current run", "sc_run_path": "//modelstation2/c$/workspace/sc_9_15_26_pnr_test/soundcast", "survey_directories": {"survey": "R:/e2projects_two/2023_base_year/2023_survey/activitysim_format/skims_attached"}, "comparison_runs_list": {"2050 new transit, old network": "\\\\modelstation3\\c$\\Workspace\\sc_new_2050_transit\\soundcast", "2050 urbansim": "\\\\modelstation2\\c$\\Workspace\\sc_2050_urbansim2_07_30_25"}, "county_map": {"33": "King", "35": "Kitsap", "53": "Pierce", "61": "Snohomish"}, "uc_list": ["@sov_inc1", "@sov_inc2", "@sov_inc3", "@hov2_inc1", "@hov2_inc2", "@hov2_inc3", "@hov3_inc1", "@hov3_inc2", "@hov3_inc3", "@av_sov_inc1", "@av_sov_inc2", "@av_sov_inc3", "@av_hov2_inc1", "@av_hov2_inc2", "@av_hov2_inc3", "@av_hov3_inc1", "@av_hov3_inc2", "@av_hov3_inc3", "@tnc_inc1", "@tnc_inc2", "@tnc_inc3", "@mveh", "@hveh", "@bveh"], "mode_list": ["sov_inc2", "hov2_inc2", "hov3_inc2", "walk", "bike", "trnst", "ferry", "commuter_rail", "litrat"], "agency_lookup": {"1": "King County Metro", "2": "Pierce Transit", "3": "Community Transit", "4": "Kitsap Transit", "5": "Washington Ferries", "6": "Sound Transit", "7": "Everett Transit"}, "emissions_scenario": "standard", "tot_veh_model_base_year": 3185281, "speed_bins": [-999999.0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, 999999.0], "fac_type_lookup": {"0": 0, "1": 4, "2": 4, "3": 5, "4": 5, "5": 5, "6": 3, "7": 5, "8": 0}, "tod_lookup": {"5to9": 5, "9to15": 9, "15to18": 15, "18to20": 18, "20to5": 20}, "summer_list": [87], "special_route_lookup": {"1671": "A-Line Rapid Ride", "1672": "B-Line Rapid Ride", "1673": "C-Line Rapid Ride", "1674": "D-Line Rapid Ride", "1675": "E-Line Rapid Ride", "1677": "H-Line Rapid Ride", "4950": "Central Link", "6995": "Tacoma Link", "6998": "Sounder South", "6999": "Sounder North", "3701": "Swift Blue Line", "3702": "Swift Green Line"}, "output_dir": "//modelstation2/c$/workspace/sc_9_15_26_pnr_test/output", "data_dir": "//modelstation2/c$/workspace/sc_9_15_26_pnr_test/data"}
input_config = {"debug_skims_and_paths": False, "model_year": "2023", "base_year": "2023", "landuse_inputs": "23_on_23_v3", "network_inputs": "network_2023_asim", "db_name": "soundcast_inputs_2023.db", "soundcast_inputs_dir": "R:/e2projects_two/SoundCast/Inputs/activitysim", "abm_model": "activitysim", "uv_directory": "C:/workspace/uv_envs/activitysim", "run_accessibility_calcs": True, "run_setup_emme_project_folders": True, "run_setup_emme_bank_folders": True, "run_copy_scenario_inputs": True, "run_import_networks": True, "run_skims_and_paths_free_flow": True, "run_skims_and_paths": True, "run_truck_model": True, "run_supplemental_trips": True, "run_abm": True, "run_summaries": True, "include_av": False, "include_tnc": True, "tnc_av": False, "include_tnc_to_transit": False, "include_knr_to_transit": False, "include_telecommute": True, "use_truck_tables": False, "run_integrated": False, "should_build_shadow_price": False, "delete_banks": False, "include_tnc_emissions": False, "add_distance_pricing": False, "distance_rate_dict": {"am": 13.5, "md": 8.5, "pm": 13.5, "ev": 8.5, "ni": 8.5}}


In [2]:
import pandas as pd
import polars as pl
import geopandas as gpd
import math
from pathlib import Path
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

# table format
pd.options.display.float_format = '{:0,.0f}'.format
format_percent = "{:,.1%}".format

In [3]:
# read data
tour_data = util.get_tour_data(summary_config)
# trip_data = util.get_trip_data(summary_config)
land_use = util.get_landuse_data(summary_config)

In [4]:
# get all drive to transit tours
tour_data_pnr = tour_data[tour_data["tour_mode"] == "DRIVE_TRN"].\
    merge(land_use[["zone_id", "TAZ", "PNR_SPACES"]], left_on="pnr_zone_id", right_on="zone_id", how="left").\
    rename(columns={'TAZ': 'pnr_taz'})

In [5]:
pnr_capacity = pd.read_csv(Path(summary_config['sc_run_path'])/ "inputs/scenario/networks/p_r_nodes.csv")
pnr_lots = util.read_sqlite_db(input_config, summary_config, "SELECT * FROM park_and_ride_lots")
pnr_landuse = land_use[land_use['PNR_SPACES']>0].\
    merge(pnr_capacity, left_on='TAZ', right_on='ZoneID', how='left').\
    merge(pnr_lots, left_on='TAZ', right_on='id', how='left')

In [6]:
# get park and ride occupancy
pnr_observed = util.read_sqlite_db(input_config, summary_config, "SELECT * FROM observed_park_and_ride WHERE data_year = 2023")

# transform to geodataframe
gdf_pnr_landuse = gpd.GeoDataFrame(
    pnr_landuse, geometry=gpd.points_from_xy(pnr_landuse.XCoord, pnr_landuse.YCoord), crs="EPSG:2285"
)

gdf_pnr_observed = gpd.GeoDataFrame(
    pnr_observed, geometry=gpd.points_from_xy(pnr_observed.x_coord, pnr_observed.y_coord), crs="EPSG:2285"
)

# merge by nearest location
gdf_merge = gdf_pnr_observed.sjoin_nearest(gdf_pnr_landuse, max_distance=500)

In [7]:
# sum occupancy across all assigned lots 
df_merge = pd.DataFrame(gdf_merge.drop(columns='geometry')).\
    groupby(['id', 'name'])[['occupancy', 'capacity', 'lot_name']].sum().reset_index()

pnr_occupancy = pnr_landuse.merge(df_merge, on=['id', 'name'], how='outer')
# TODO: how was the network park and ride made? capacity mode accurate in network than in observed
pnr_occupancy['cap_diff'] = pnr_occupancy['capacity']-pnr_occupancy['Capacity']

In [8]:
# calculate park and ride model baordings and merge with occupancy data
pnr_data = tour_data_pnr[['pnr_taz','PNR_SPACES']].value_counts().reset_index().\
    merge(pnr_occupancy[['ZoneID',"name", 'occupancy']], left_on='pnr_taz', right_on='ZoneID', how='outer').\
    rename(columns={'count': 'model occupancy', 
                    'occupancy': 'observed occupancy',
                    'PNR_SPACES': 'PNR capacity'})

pnr_data['observed occupancy rate'] = pnr_data['observed occupancy']/pnr_data['PNR capacity']
pnr_data['model occupancy rate'] = pnr_data['model occupancy']/pnr_data['PNR capacity']
pnr_data['percent diff'] = (pnr_data['model occupancy']-pnr_data['observed occupancy']) / pnr_data['observed occupancy']

pnr_data = pnr_data[(~pnr_data['observed occupancy rate'].isna()) | (~pnr_data['model occupancy rate'].isna())]

- percent difference = (model occupancy - observed occupancy) / observed occupancy

## Occupancy v.s. Capacity

In [9]:
def plot_occupancy(data, x, y, title):

    p_max = data['percent diff'].max()
    p_min = data['percent diff'].min()
    p_midpoint = (0-p_min) / (p_max-p_min)
    c_max = data['PNR capacity'].max()

    fig = px.scatter(data, x=x, y=y, 
                    color='percent diff', 
                    color_continuous_scale=[(0,"#00716c"),(p_midpoint, "#E3C9E3"),(1, "#4a0048")],
                    range_color=[p_min,p_max],
                    hover_name='name',
                    hover_data={'model occupancy rate':':.2%',
                    
                                'percent diff':':.2%'},
                    title=title)


    fig.add_shape(type='line',
                x0=0,y0=0,x1=c_max,y1=c_max,
                line=dict(width=1, dash='dash'))
    fig.update_xaxes(dtick=250,range=[0, c_max])
    fig.update_yaxes(dtick=250,range=[0, c_max])
    fig.update_traces(marker_size=5)
    fig.update_layout(height=650, width=700, font=dict(size=11))
    fig.show()

plot_occupancy(pnr_data, x="PNR capacity", y="model occupancy", title="Park and Ride: Capacity v.s. Model Occupancy")

In [10]:
gdf_plot = gdf_pnr_landuse[['name','ZoneID','geometry']].\
    merge(pnr_data[['pnr_taz','PNR capacity', 'model occupancy', 'observed occupancy', 'observed occupancy rate',
       'model occupancy rate']], left_on='ZoneID', right_on='pnr_taz', how='left')
gdf_plot.to_crs('EPSG:4326', inplace=True)
gdf_plot['latitude'] = gdf_plot.geometry.y
gdf_plot['longitude'] = gdf_plot.geometry.x

fig = px.scatter_map(gdf_plot, 
                     lat='latitude', lon='longitude', 
                     color='model occupancy rate', 
                     color_continuous_scale=["#E3C9E3", "#AD5CAB", "#4a0048"],
                     hover_name='name',
                     hover_data={'latitude':False,
                                 'longitude':False,
                                 'ZoneID':':.0f',
                                 'model occupancy': ':.0f', 
                                 'PNR capacity': ':.0f', 
                                 'model occupancy rate': ':.2%'}, 
                     title='Park and Ride Occupancy Rate')
# fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.update_traces(marker_size=8)
fig.update_layout(height=750, width=850, font=dict(size=11))

fig.show()

## Observed v.s. Model Occupancy

In [11]:
plot_occupancy(pnr_data, x="observed occupancy", y="model occupancy", title="Park and Ride: Observed v.s. Model Occupancy")

- park and ride occupancy table

In [12]:
df_table = pnr_data[['pnr_taz', 'name', 'PNR capacity', 'observed occupancy', 'model occupancy', 
               'observed occupancy rate', 'model occupancy rate','percent diff']].\
    style.format({'pnr_taz': '{:.0f}',
                  'PNR capacity': '{:,.0f}',
                  'observed occupancy': '{:,.0f}', 
                  'model occupancy': '{:,.0f}',
                  'observed occupancy rate': '{:.1%}',
                  'model occupancy rate': '{:.1%}',
                  'percent diff': '{:.1%}'
})
df_table
# import itables
# itables.show(df_table,allow_html=True, paging=False, scrollY="550px", scrollCollapse=True, style="width: auto;")

,pnr_taz,name,PNR capacity,observed occupancy,model occupancy,observed occupancy rate,model occupancy rate,percent diff
0,3718,nan,390,nan,260,nan%,66.7%,nan%
1,3719,nan,"2,283",nan,911,nan%,39.9%,nan%
2,3720,nan,220,nan,28,nan%,12.7%,nan%
3,3721,nan,"1,067",nan,556,nan%,52.1%,nan%
4,3722,nan,302,nan,119,nan%,39.4%,nan%
5,3723,nan,63,nan,7,nan%,11.1%,nan%
6,3724,nan,996,nan,600,nan%,60.2%,nan%
7,3725,nan,636,nan,556,nan%,87.4%,nan%
8,3726,nan,600,nan,46,nan%,7.7%,nan%
9,3727,nan,104,nan,36,nan%,34.6%,nan%


In [13]:
gdf_plot = gdf_pnr_landuse[['name','ZoneID','geometry']].\
    merge(pnr_data[['pnr_taz','PNR capacity', 'model occupancy', 'observed occupancy', 'observed occupancy rate',
       'model occupancy rate','percent diff']], left_on='ZoneID', right_on='pnr_taz', how='left')
gdf_plot.to_crs('EPSG:4326', inplace=True)
gdf_plot['latitude'] = gdf_plot.geometry.y
gdf_plot['longitude'] = gdf_plot.geometry.x

p_max = gdf_plot['percent diff'].max()
p_min = gdf_plot['percent diff'].min()
p_midpoint = (0-p_min) / (p_max-p_min)
c_max = gdf_plot['PNR capacity'].max()

fig = px.scatter_map(gdf_plot, 
                     lat='latitude', lon='longitude', 
                     color='percent diff', 
                    color_continuous_scale=[(0,"#00716c"),(p_midpoint, "#E3C9E3"),(1, "#4a0048")],
                    range_color=[p_min,p_max],
                     hover_name='name',
                     hover_data={'latitude':False,
                                 'longitude':False,
                                 'ZoneID':':.0f',
                                 'model occupancy': ':.0f', 
                                 'PNR capacity': ':.0f', 
                                 'percent diff': ':.2%'}, 
                     title='Park and Ride Observed and Model Occupancy Percent Difference')
# fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.update_traces(marker_size=8)
fig.update_layout(height=750, width=850, font=dict(size=11))
fig.show()